In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import stft, get_window, spectrogram
import os
from sigmf import SigMFFile
import sigmf as sig
import json
from skimage.transform import resize
import matplotlib.pyplot as plt
import struct

# Panoradio

In [ ]:
def save_spectogram(file_path, start_vector=0, num=0):
    data = np.load(file_path)

    num_vectors = 20
    signal = data[start_vector:start_vector + num_vectors].flatten()

    desired_shape=(256, 256)
    window_size = 512
    overlap = 0
    fft_size = 1024
    window = get_window('hamming', window_size)

    _, _, Sxx = spectrogram(
        signal,
        window=window,
        nperseg=window_size,
        noverlap=overlap,
        nfft=fft_size,
        scaling='density',
        mode='magnitude'
    )

    # Convert to dB scale
    Sxx = np.fft.fftshift(Sxx, axes=0)
    Sxx_dB = 10 * np.log10(Sxx + 1e-10)

    S_resized = resize(Sxx_dB, desired_shape, mode='reflect', anti_aliasing=True)

    save_dir = r"dataset_panoradio"
    filename = f"spectogram_panoradio_{num}.png"
    os.makedirs(save_dir, exist_ok=True)  
    filepath = os.path.join(save_dir, filename)
    
    plt.imsave(filepath, S_resized, cmap='viridis')
    print(f"Saved {filename}")

In [ ]:
# path to panoradio dataset file
file_path = 'dataset_hf_radio.npy'

In [ ]:
num = 0
for i in range(0,172800,20):
    save_spectogram(file_path, i, num)
    num += 1

# ICARUS

In [ ]:
def save_spectogram(signal, num):

    desired_shape=(256, 256)
    window_size = 512
    overlap = 0
    fft_size = 1024  
    window = get_window('hamming', window_size)

    _, _, Sxx = spectrogram(
        signal,
        window=window,
        nperseg=window_size,
        noverlap=overlap,
        nfft=fft_size,
        scaling='density',
        mode='magnitude'
    )

    Sxx = np.fft.fftshift(Sxx, axes=0)
    Sxx_dB = 10 * np.log10(Sxx + 1e-10)

    S_resized = resize(Sxx_dB, desired_shape, mode='reflect', anti_aliasing=True)

    save_dir = r"dataset_POWDER_10M"
    filename = f"spectogram_POWDER_{num}.png"
    os.makedirs(save_dir, exist_ok=True) 
    filepath = os.path.join(save_dir, filename)
    
    plt.imsave(filepath, S_resized, cmap='viridis')
    print(f"Saved {filename}")

In [ ]:
# path to folder of files of ICARUS dataset
folder_path = 'neu_4f18jp909/POWDER_Dataset/Batch1_10MHz/IQ/'

num = 0
for filename in os.listdir(folder_path):

    samples = np.fromfile(folder_path+filename, dtype=np.complex64)
    
    chunk_size = 40960
    for i in range(0, len(samples), chunk_size):
        chunk = samples[i:i + chunk_size]
        if len(chunk) == chunk_size:
            save_spectogram(chunk, num)
            num += 1
    if num == 15000:
        break

# POWDER

In [ ]:
def save_spectogram(signal, num):

    desired_shape=(256, 256)
    window_size = 512
    overlap = 0
    fft_size = 1024  
    window = get_window('hamming', window_size)

    _, _, Sxx = spectrogram(
        signal,
        window=window,
        nperseg=window_size,
        noverlap=overlap,
        nfft=fft_size,
        scaling='density',
        mode='magnitude'
    )

    # Convert to dB scale
    Sxx = np.fft.fftshift(Sxx, axes=0)
    Sxx_dB = 10 * np.log10(Sxx + 1e-10)

    S_resized = resize(Sxx_dB, desired_shape, mode='reflect', anti_aliasing=True)

    save_dir = r"dataset_Globe_POWDER"
    filename = f"spectogram_Globe_POWDER_{num}.png"
    os.makedirs(save_dir, exist_ok=True)  
    filepath = os.path.join(save_dir, filename)
    
    plt.imsave(filepath, S_resized, cmap='viridis')
    print(f"Saved {filename}")

In [ ]:
# path to folder of files of POWDER dataset
folder_path = 'neu_m046tb444/GlobecomPOWDER/'

num = 0
for filename in os.listdir(folder_path):
    if filename.endswith('.bin'):
        
        samples = np.fromfile(folder_path+filename, dtype=np.complex64)
        
        chunk_size = 40960
        for i in range(0, len(samples), chunk_size):
            chunk = samples[i:i + chunk_size]
            if len(chunk) == chunk_size:
                save_spectogram(chunk, num)
                num += 1
        if num == 15000:
            break

# Real world LTE and Radar signals

In [ ]:
# path to folder fo files of LTE dataset
folder_path = '15M/'
sigmf_to_numpy = {
    'cf32_le': np.complex64,
    'cf64_le': np.complex128,
    'i16_le': np.int16,
    'u8': np.uint8,
}

In [ ]:
num = 0
for filename in os.listdir(folder_path):
    if filename.endswith('.sigmf-data'):
        data_path = os.path.join(folder_path, filename)
        meta_path = data_path.replace('.sigmf-data', '.sigmf-meta')

        if not os.path.exists(meta_path):
            print(f"Skipping {filename} — no matching .sigmf-meta file.")
            continue

        # Load metadata
        with open(meta_path, 'r') as f:
            metadata = json.load(f)

        # Get datatype
        datatype_str = metadata["global"].get("core:datatype")
        if datatype_str not in sigmf_to_numpy:
            print(f"Unsupported datatype '{datatype_str}' in {filename}")
            continue

        np_dtype = sigmf_to_numpy[datatype_str]

        samples = np.fromfile(data_path, dtype=np_dtype)
        print(f"{filename}: loaded {len(samples)} samples of type {np_dtype}")
        
        chunk_size = 40960
        for i in range(0, len(samples), chunk_size):
            chunk = samples[i:i + chunk_size]
            if len(chunk) == chunk_size:
                save_spectogram(chunk, num)
                num += 1
        if num == 15000:
            break